In [1]:
!pip install -q transformers==4.47.0 accelerate bitsandbytes sentencepiece
!pip install -q opencv-python-headless Pillow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 62.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.1 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which i

In [2]:
import os
import json
import base64
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch
import warnings
warnings.filterwarnings("ignore")

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME = "LanguageBind/Video-LLaVA-7B-hf"
MAX_FRAMES_PER_CLIP = 8   # T4 memory limit — keep at 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Paths (update these to match your Kaggle dataset paths) ──────────────────
DATASET_JSON = "/kaggle/input/real-4dbench-data/dataset.json"
RGB_FRAMES_DIR = "/kaggle/input/real-4dbench-data/rgb_frames"
OUTPUT_PATH = "/kaggle/working/videollava_rgb_results.csv"

print(f"Device: {DEVICE}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
from transformers import AutoProcessor, LlavaNextVideoForConditionalGeneration
MODEL_NAME = "llava-hf/LLaVA-NeXT-Video-7B-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaNextVideoForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True
)
print("Model loaded successfully.")

2026-06-06 13:09:10.974088: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780751351.203937      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780751351.270338      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780751351.813631      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780751351.813668      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780751351.813671      58 computation_placer.cc:177] computation placer alr

processor_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded successfully.


In [5]:
def load_frames_for_clip(frames_dir, clip_id, max_frames=MAX_FRAMES_PER_CLIP):
    """
    Load RGB frames for a clip as PIL Images.
    Video-LLaVA expects PIL Image objects, not base64.
    """
    clip_dir = os.path.join(frames_dir, clip_id)

    if not os.path.exists(clip_dir):
        print(f"WARNING: No frames directory found for {clip_id}")
        return []

    frame_files = sorted([
        f for f in os.listdir(clip_dir)
        if f.endswith('.jpg') or f.endswith('.png')
    ])

    if not frame_files:
        print(f"WARNING: No frames found in {clip_dir}")
        return []

    # Select evenly spaced frames
    if len(frame_files) > max_frames:
        step = len(frame_files) // max_frames
        frame_files = frame_files[::step][:max_frames]

    frames = []
    for filename in frame_files:
        frame_path = os.path.join(clip_dir, filename)
        img = Image.open(frame_path).convert("RGB")
        # Resize to reduce memory usage
        img = img.resize((336, 336))
        frames.append(np.array(img))

    return frames


def load_frames_with_depth(rgb_dir, depth_dir, clip_id,
                            max_frames=MAX_FRAMES_PER_CLIP):
    """
    Load RGB frames alternating with depth frames.
    RGB+Depth condition.
    """
    rgb_frames = load_frames_for_clip(rgb_dir, clip_id, max_frames)

    depth_clip_dir = os.path.join(depth_dir, clip_id)
    if not os.path.exists(depth_clip_dir):
        print(f"No depth frames for {clip_id} — RGB only")
        return rgb_frames

    depth_files = sorted([
        f for f in os.listdir(depth_clip_dir)
        if f.endswith('.jpg') or f.endswith('.png')
    ])

    combined = []
    for i, rgb_frame in enumerate(rgb_frames):
        combined.append(rgb_frame)
        if i < len(depth_files):
            depth_path = os.path.join(depth_clip_dir, depth_files[i])
            depth_img = Image.open(depth_path).convert("RGB").resize((336, 336))
            combined.append(np.array(depth_img))

    # Cap at max_frames * 2
    return combined[:max_frames * 2]

print("Frame loading functions ready.")

Frame loading functions ready.


In [17]:
def build_prompt(question, options, condition="rgb"):
    options_text = "\n".join([f"({k}) {v}" for k, v in options.items()])
    if condition == "rgb":
        context = "You are watching a sequence of video frames showing a physical occlusion event."
    else:
        context = (
            "You are watching a sequence of video frames showing a physical occlusion event. "
            "Every second frame is a colorized depth map where colors represent distances "
            "(blue = close, red = far). Use both RGB and depth information to answer."
        )
    return f"""{context}

Question: {question}

Options:
{options_text}

Answer with ONLY the letter A, B, C, or D. Nothing else.
Your answer:"""


def evaluate_single_qa(frames, question, options, condition="rgb"):
    if not frames:
        return "ERROR"

    prompt = build_prompt(question, options, condition)
    full_prompt = f"USER: <video>\n{prompt}\nASSISTANT:"

    try:
        video = np.stack(frames, axis=0)

        inputs = processor(
            text=full_prompt,
            videos=[video],
            return_tensors="pt",
            padding=True
        ).to(DEVICE)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                temperature=None,
                top_p=None
            )

        generated = output[0][inputs["input_ids"].shape[1]:]
        answer_text = processor.decode(generated, skip_special_tokens=True).strip().upper()

        for letter in ["A", "B", "C", "D"]:
            if letter in answer_text:
                return letter

        return "INVALID"

    except Exception as e:
        print(f"Inference error: {e}")
        torch.cuda.empty_cache()
        return "ERROR"

print("Inference functions ready.")

Inference functions ready.


In [18]:
def evaluate_dataset(dataset_path, rgb_frames_dir, output_path, condition="rgb"):
    with open(dataset_path, "r") as f:
        dataset = json.load(f)

    results = []
    total_clips = len(dataset["clips"])
    print(f"Evaluating {total_clips} clips — Condition: {condition}")
    print(f"Model: Video-LLaVA-7B\n")

    for i, clip in enumerate(tqdm(dataset["clips"], desc="Clips")):
        clip_id = clip["clip_id"]

        frames = load_frames_for_clip(rgb_frames_dir, clip_id)

        if not frames:
            print(f"Skipping {clip_id} — no frames")
            continue

        for qa in clip["qa_pairs"]:
            model_answer = evaluate_single_qa(
                frames,
                qa["question"],
                qa["options"],
                condition
            )

            results.append({
                "clip_id": clip_id,
                "question_id": qa["question_id"],
                "question_type": qa["question_type"],
                "question": qa["question"],
                "correct_answer": qa["correct_answer"],
                "model_answer": model_answer,
                "is_correct": model_answer == qa["correct_answer"],
                "condition": condition,
                "model": "video-llava",
                "object_type": clip["object_type"],
                "occluder_type": clip["occluder_type"],
                "scenario_type": clip["scenario_type"]
            })

        # Clear GPU cache every 10 clips
        if (i + 1) % 10 == 0:
            torch.cuda.empty_cache()
            df = pd.DataFrame(results)
            df.to_csv(output_path, index=False)
            accuracy = df["is_correct"].mean() * 100
            print(f"Checkpoint — {i+1}/{total_clips} clips | Accuracy so far: {accuracy:.1f}%")

    # Final save
    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False)

    accuracy = df["is_correct"].mean() * 100
    print(f"\nDone. Total questions: {len(results)}")
    print(f"Overall Accuracy: {accuracy:.1f}%")
    print(f"Results saved to {output_path}")

    return df

print("Evaluation loop ready.")

Evaluation loop ready.


In [19]:
# Run RGB condition first
DATASET_JSON = "/kaggle/input/datasets/owaisahmad9870/real-4dbench-data/dataset.json"
RGB_FRAMES_DIR = "/kaggle/input/datasets/owaisahmad9870/real-4dbench-data/rgb_frames"
df_results = evaluate_dataset(
    dataset_path=DATASET_JSON,
    rgb_frames_dir=RGB_FRAMES_DIR,
    output_path=OUTPUT_PATH,
    condition="rgb"
)

# Print summary
print("\n── Summary ───────────────────────────────────────")
print(f"Total questions: {len(df_results)}")
print(f"Correct: {df_results['is_correct'].sum()}")
print(f"Overall Accuracy: {df_results['is_correct'].mean()*100:.1f}%")
print("\nAccuracy by question type:")
print(df_results.groupby("question_type")["is_correct"].mean().mul(100).round(1))
print("\nAccuracy by scenario:")
print(df_results.groupby("scenario_type")["is_correct"].mean().mul(100).round(1))

Evaluating 1 clips — Condition: rgb
Model: Video-LLaVA-7B



Clips: 100%|██████████| 1/1 [00:15<00:00, 15.37s/it]


Done. Total questions: 2
Overall Accuracy: 50.0%
Results saved to /kaggle/working/videollava_rgb_results.csv

── Summary ───────────────────────────────────────
Total questions: 2
Correct: 1
Overall Accuracy: 50.0%

Accuracy by question type:
question_type
state       100.0
tracking      0.0
Name: is_correct, dtype: float64

Accuracy by scenario:
scenario_type
full_occlusion    50.0
Name: is_correct, dtype: float64


In [20]:
# This saves results to Kaggle output
# Download from Kaggle output panel after run completes
print(f"Results file location: {OUTPUT_PATH}")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1024:.1f} KB")
print("\nDownload this file from the Kaggle output panel (right sidebar)")
print("Then place it in your local results/raw_outputs/ folder")

Results file location: /kaggle/working/videollava_rgb_results.csv
File size: 0.4 KB

Download this file from the Kaggle output panel (right sidebar)
Then place it in your local results/raw_outputs/ folder
